# Coastal flood step 05: expected annual damage calculations (minimum_scenario)

- Computes asset/subsector/sector expected annual damages (USD).
- Saves CSV outputs used by later map and attribution notebooks.


In [ ]:
from pathlib import Path
import sys
import importlib

import pandas

pandas.set_option('display.max_columns', 200)
pandas.set_option('display.width', 200)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# Reporting in USD only
USD_PER_JMD = 1.0 / 150.0



In [ ]:
# Core paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
results_path = base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario'
direct_damages_path = results_path / 'direct_damages'
damage_estimates_path = results_path / 'damage_estimates'
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'

if not direct_damages_path.exists():
    raise FileNotFoundError(f'Missing folder: {direct_damages_path}')
if not network_csv.exists():
    raise FileNotFoundError(f'Missing file: {network_csv}')

print(f'direct_damages_path: {direct_damages_path}')
print(f'network_csv: {network_csv}')



In [ ]:
# Discover all direct-damage parquet files produced for sensitivity parameter set 0
parquet_files = sorted(direct_damages_path.rglob('*_direct_damages_parameter_set_0.parquet'))

if not parquet_files:
    raise FileNotFoundError(f'No direct damage parquet files found under {direct_damages_path}')

print(f'Found {len(parquet_files)} direct-damage files.')
pandas.DataFrame({'parquet_file': [str(p) for p in parquet_files]})


In [ ]:
# Read network metadata and build expected file mapping (asset/layer/id column)
network_details = pandas.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_details = network_details[required_cols].drop_duplicates().copy()
network_details['folder_name'] = network_details['asset_gpkg'] + '_' + network_details['asset_layer']
network_details['expected_parquet'] = network_details['folder_name'].apply(
    lambda folder: direct_damages_path / folder / f'{folder}_direct_damages_parameter_set_0.parquet'
)
network_details['exists'] = network_details['expected_parquet'].apply(lambda p: p.exists())

display(network_details[['sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column', 'exists']].sort_values(['sector', 'asset_gpkg', 'asset_layer']))

missing_files = network_details.loc[~network_details['exists'], ['asset_gpkg', 'asset_layer', 'expected_parquet']]
if len(missing_files) > 0:
    print('Missing expected files:')
    display(missing_files)
else:
    print('All expected direct-damage files are present.')


In [ ]:
# Load each damage table with the key columns needed for coastal EAD setup
required_damage_cols = [
    'coastal_flood_fn_mg_rp_25',
    'coastal_flood_fn_mg_rp_100',
    'coastal_flood_fn_mg_rp_500',
    'coastal_flood_fn_nomg_rp_25',
    'coastal_flood_fn_nomg_rp_100',
    'coastal_flood_fn_nomg_rp_500',
]

loaded_damage_tables = {}
load_summary_rows = []

for row in network_details.itertuples(index=False):
    parquet_path = row.expected_parquet
    if not parquet_path.exists():
        continue

    damage_df = pandas.read_parquet(parquet_path)

    missing = [c for c in required_damage_cols if c not in damage_df.columns]
    key = f'{row.asset_gpkg}_{row.asset_layer}'
    loaded_damage_tables[key] = damage_df

    load_summary_rows.append({
        'table_key': key,
        'sector': row.sector,
        'subsector': row.asset_description,
        'rows': len(damage_df),
        'columns': len(damage_df.columns),
        'missing_required_cols': ', '.join(missing) if missing else '',
        'parquet_path': str(parquet_path),
    })

load_summary = pandas.DataFrame(load_summary_rows).sort_values(['sector', 'table_key']).reset_index(drop=True)
display(load_summary)

bad_tables = load_summary[load_summary['missing_required_cols'] != '']
if len(bad_tables) > 0:
    print('Some tables are missing required coastal EAD columns:')
    display(bad_tables[['table_key', 'missing_required_cols']])
else:
    print('All loaded tables have the required RP 25/100/500 with/without mangrove columns.')


In [ ]:
# Optional preview: pick one loaded table
table_to_preview = 'roads_edges'  # change as needed

if table_to_preview not in loaded_damage_tables:
    print(f"'{table_to_preview}' not found. Available keys:")
    print(sorted(loaded_damage_tables.keys()))
else:
    preview_cols = [
        c for c in loaded_damage_tables[table_to_preview].columns
        if c.startswith('coastal_flood_fn_') and '_rp_' in c
    ]
    display(loaded_damage_tables[table_to_preview][preview_cols].head(10))


In [ ]:
# Import Robyn_river_floods library (without modifying the .py file)
robyn_lib_path = base_path / 'robyns_libraries'
if str(robyn_lib_path) not in sys.path:
    sys.path.append(str(robyn_lib_path))

import Robyn_river_floods
importlib.reload(Robyn_river_floods)

print(f'Using Robyn_river_floods from: {Robyn_river_floods.__file__}')


In [ ]:
# Compute asset-level EADs using RP25/100/500 for with-mangroves and without-mangroves (USD)
rp_mg_cols = [
    'coastal_flood_fn_mg_rp_25',
    'coastal_flood_fn_mg_rp_100',
    'coastal_flood_fn_mg_rp_500',
]
rp_nomg_cols = [
    'coastal_flood_fn_nomg_rp_25',
    'coastal_flood_fn_nomg_rp_100',
    'coastal_flood_fn_nomg_rp_500',
]

mg_to_rp = {
    'coastal_flood_fn_mg_rp_25': 'rp25.0',
    'coastal_flood_fn_mg_rp_100': 'rp100.0',
    'coastal_flood_fn_mg_rp_500': 'rp500.0',
}
nomg_to_rp = {
    'coastal_flood_fn_nomg_rp_25': 'rp25.0',
    'coastal_flood_fn_nomg_rp_100': 'rp100.0',
    'coastal_flood_fn_nomg_rp_500': 'rp500.0',
}

asset_ead_rows = []

for row in network_details.itertuples(index=False):
    table_key = f'{row.asset_gpkg}_{row.asset_layer}'
    if table_key not in loaded_damage_tables:
        continue

    damage_df = loaded_damage_tables[table_key].copy()

    needed = [row.asset_id_column] + rp_mg_cols + rp_nomg_cols
    missing = [c for c in needed if c not in damage_df.columns]
    if missing:
        print(f"Skipping {table_key}: missing columns {missing}")
        continue

    grouped = (
        damage_df[[row.asset_id_column] + rp_mg_cols + rp_nomg_cols]
        .groupby(row.asset_id_column, as_index=False)
        .sum()
        .copy()
    )

    mg_df = grouped[rp_mg_cols].rename(columns=mg_to_rp)
    nomg_df = grouped[rp_nomg_cols].rename(columns=nomg_to_rp)

    ead_with_mg_jmd = Robyn_river_floods.calculate_ead(mg_df)
    ead_without_mg_jmd = Robyn_river_floods.calculate_ead(nomg_df)

    grouped['EAD_With_Mangroves_USD'] = ead_with_mg_jmd * USD_PER_JMD
    grouped['EAD_Without_Mangroves_USD'] = ead_without_mg_jmd * USD_PER_JMD
    grouped['Avoided_EAD_USD'] = grouped['EAD_Without_Mangroves_USD'] - grouped['EAD_With_Mangroves_USD']
    grouped['Avoided_EAD_Share_of_Baseline'] = grouped['Avoided_EAD_USD'] / grouped['EAD_Without_Mangroves_USD']
    grouped.loc[grouped['EAD_Without_Mangroves_USD'] <= 0, 'Avoided_EAD_Share_of_Baseline'] = pandas.NA

    grouped['Sector'] = row.sector
    grouped['Subsector'] = row.asset_description
    grouped['Asset'] = row.asset_gpkg
    grouped['Layer'] = row.asset_layer
    grouped = grouped.rename(columns={row.asset_id_column: 'Asset_ID'})

    out_cols = [
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD', 'Avoided_EAD_Share_of_Baseline'
    ]
    asset_ead_rows.append(grouped[out_cols])

asset_ead = pandas.concat(asset_ead_rows, ignore_index=True) if asset_ead_rows else pandas.DataFrame()

print(f'Asset rows with EAD (USD): {len(asset_ead):,}')
display(asset_ead.head(20))


In [ ]:
# Summaries: subsector and sector EAD (USD)
if asset_ead.empty:
    raise ValueError('asset_ead is empty; check missing columns or source files.')

subsector_ead_summary = (
    asset_ead
    .groupby(['Sector', 'Subsector'], as_index=False)[
        ['EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD']
    ]
    .sum()
)
subsector_ead_summary['Avoided_EAD_Share_of_Baseline'] = (
    subsector_ead_summary['Avoided_EAD_USD'] / subsector_ead_summary['EAD_Without_Mangroves_USD']
)
subsector_ead_summary.loc[
    subsector_ead_summary['EAD_Without_Mangroves_USD'] <= 0,
    'Avoided_EAD_Share_of_Baseline'
] = pandas.NA

sector_ead_summary = (
    asset_ead
    .groupby(['Sector'], as_index=False)[
        ['EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD']
    ]
    .sum()
)
sector_ead_summary['Avoided_EAD_Share_of_Baseline'] = (
    sector_ead_summary['Avoided_EAD_USD'] / sector_ead_summary['EAD_Without_Mangroves_USD']
)
sector_ead_summary.loc[
    sector_ead_summary['EAD_Without_Mangroves_USD'] <= 0,
    'Avoided_EAD_Share_of_Baseline'
] = pandas.NA

print('Subsector EAD summary (USD):')
display(subsector_ead_summary.sort_values(['Sector', 'Subsector']))

print('Sector EAD summary (USD):')
display(sector_ead_summary.sort_values(['Sector']))


In [ ]:
# Save outputs (USD only)
damage_estimates_path.mkdir(parents=True, exist_ok=True)

asset_out = damage_estimates_path / 'coastal_ead_asset_level_usd.csv'
subsector_out = damage_estimates_path / 'coastal_ead_subsector_summary_usd.csv'
sector_out = damage_estimates_path / 'coastal_ead_sector_summary_usd.csv'

asset_ead.to_csv(asset_out, index=False)
subsector_ead_summary.to_csv(subsector_out, index=False)
sector_ead_summary.to_csv(sector_out, index=False)

print(f'Saved: {asset_out}')
print(f'Saved: {subsector_out}')
print(f'Saved: {sector_out}')



In [ ]:
# USD output preview
print('Asset-level EAD in USD:')
display(asset_ead.head(20))

print('Subsector EAD summary in USD:')
display(subsector_ead_summary.sort_values(['Sector', 'Subsector']))

print('Sector EAD summary in USD:')
display(sector_ead_summary.sort_values(['Sector']))


In [ ]:
# Percentage of avoided EAD relative to no-mangroves EAD (USD)
# Formula: % avoided = 100 * Avoided_EAD / EAD_Without_Mangroves
subsector_pct = subsector_ead_summary.copy()
sector_pct = sector_ead_summary.copy()

for dataframe in [subsector_pct, sector_pct]:
    dataframe['Percent_Avoided_EAD_vs_NoMangroves'] = 100.0 * dataframe['Avoided_EAD_Share_of_Baseline']

print('Sector-level percentage avoided EAD (% of no-mangroves EAD):')
display(
    sector_pct[['Sector', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD', 'Percent_Avoided_EAD_vs_NoMangroves']]
    .sort_values('Sector')
    .round({'Percent_Avoided_EAD_vs_NoMangroves': 2})
)

print('Subsector-level percentage avoided EAD (% of no-mangroves EAD):')
display(
    subsector_pct[['Sector', 'Subsector', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD', 'Percent_Avoided_EAD_vs_NoMangroves']]
    .sort_values(['Sector', 'Subsector'])
    .round({'Percent_Avoided_EAD_vs_NoMangroves': 2})
)

sector_pct_out = damage_estimates_path / 'coastal_ead_sector_summary_usd_with_pct_avoided.csv'
subsector_pct_out = damage_estimates_path / 'coastal_ead_subsector_summary_usd_with_pct_avoided.csv'

sector_pct.to_csv(sector_pct_out, index=False)
subsector_pct.to_csv(subsector_pct_out, index=False)

print(f'Saved: {sector_pct_out}')
print(f'Saved: {subsector_pct_out}')

